<a href="https://colab.research.google.com/github/CemKeremSahin/NIR-to-RGB-Colorization/blob/main/Notebooks/Vanilla_Autoencoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
print(f"Sahip olduğun CPU çekirdek sayısı: {os.cpu_count()}")

In [ ]:
# --- 1. COLAB KURULUMLARI VE DRIVE BAĞLANTISI ---
import os

# Gerekli kütüphaneyi kuruyoruz
try:
    import segmentation_models_pytorch as smp
    print("Kütüphane zaten kurulu.")
except ImportError:
    print("Kütüphane kuruluyor...")
    os.system('pip install segmentation_models_pytorch')
    import segmentation_models_pytorch as smp

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Drive dan colab e aktarma
!cp -r /content/drive/MyDrive/realword_dataset/train /content/

In [ ]:
import os
# Ayarladığınız yolu buraya yazın
NIR_FOLDER = "/content/train/NIR"
RGB_FOLDER = "/content/train/RGB"

print("--- VERİ SETİ KONTROLÜ BAŞLIYOR ---\n")

def veri_kontrol():
    # NIR Klasörünü kontrol et
    if not os.path.exists(NIR_FOLDER):
        print(f"HATA: {NIR_FOLDER} klasörü bulunamadı! Lütfen '!cp' komutunu kontrol et.")
        return

    # Sadece .bmp dosyalarını listele ve sırala
    nir_list = sorted([f for f in os.listdir(NIR_FOLDER) if f.lower().endswith('.bmp')])
    print(f"NIR klasöründe {len(nir_list)} adet .bmp dosyası bulundu.")

    # RGB Klasörünü kontrol et (Eğer varsa)
    if os.path.exists(RGB_FOLDER):
        rgb_list = sorted([f for f in os.listdir(RGB_FOLDER) if f.lower().endswith('.bmp')])
        print(f"RGB klasöründe {len(rgb_list)} adet .bmp dosyası bulundu.")

        # Makale verisiyle karşılaştırma
        if len(nir_list) == 538:
            print(">>> DURUM: Makaledeki 'Real-World' veri seti sayısına (538) tam uyuyor! [cite: 387]")

        # Eşleşme kontrolü
        if len(nir_list) == len(rgb_list):
            print(">>> Başarılı: NIR ve RGB dosya sayıları eşit.")
        else:
            print(f">>> UYARI: Dosya sayıları farklı! (NIR: {len(nir_list)}, RGB: {len(rgb_list)})")

        print("\nÖrnek Eşleşmeler (İlk 5):")
        for i in range(min(5, len(nir_list))):
            print(f"  {nir_list[i]} <--*--> {rgb_list[i] if i < len(rgb_list) else 'Eksik!'}")
    else:
        print(f"NOT: {RGB_FOLDER} klasörü bulunamadı. Sadece NIR dosyaları işlenebilir.")

veri_kontrol()

--- VERİ SETİ KONTROLÜ BAŞLIYOR ---

NIR klasöründe 458 adet .bmp dosyası bulundu.
RGB klasöründe 458 adet .bmp dosyası bulundu.
>>> Başarılı: NIR ve RGB dosya sayıları eşit.

Örnek Eşleşmeler (İlk 5):
  0001.bmp <--*--> 0001.bmp
  0002.bmp <--*--> 0002.bmp
  0003.bmp <--*--> 0003.bmp
  0004.bmp <--*--> 0004.bmp
  0005.bmp <--*--> 0005.bmp


In [ ]:
import os
import cv2
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models
from torchvision.transforms import Normalize
from tqdm import tqdm

# --- 1. A100 GPU SÜPER GÜÇLERİ (TF32 AKTİF) ---
torch.backends.cudnn.allow_tf32 = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Kullanılan Cihaz: {DEVICE}")

# --- 2. AYARLAR VE YOLLAR ---
ROOT_PATH = "/content/train"
SAVE_PATH = "/content/drive/MyDrive/Vanilla_Autoencoder_Realword_Dataset/Weights"
os.makedirs(SAVE_PATH, exist_ok=True)

PATCH_SIZE = 256
BATCH_SIZE = 16    # Makalede 5 kullanılmıştı, ancak A100 için 16 ideal
NUM_EPOCHS = 3500   # Makaledeki değer
LEARNING_RATE = 2e-4 # Makaledeki değer
SAVE_EVERY_N_EPOCHS = 500

# --- 3. VANILLA AUTOENCODER MİMARİSİ (SKIP CONNECTIONS KALDIRILDI) ---
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2, inplace=True)
        )
    def forward(self, x):
        return self.double_conv(x)

class VanillaAutoencoder(nn.Module):
    def __init__(self, in_channels=1, out_channels=3):
        super().__init__()
        # Encoder
        self.enc1 = DoubleConv(in_channels, 64)
        self.enc2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.enc3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.enc4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))

        # Bottleneck (Darboğaz)
        self.bottleneck = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 1024))

        # Decoder (Skip bağlantıları iptal edildiği için kanal sayıları direkt UpSample'dan gelenle aynı kalır)
        self.up1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(512, 512)

        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(256, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(128, 128)

        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(64, 64)

        self.outc = nn.Sequential(
            nn.Conv2d(64, out_channels, kernel_size=1),
            nn.Tanh() # Görüntü aralığını [-1, 1] yapar
        )

    def forward(self, x):
        # Encoder Akışı
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)

        b = self.bottleneck(e4)

        # Decoder Akışı (Doğrusal akış, torch.cat yok!)
        d1 = self.up1(b)
        d1 = self.dec1(d1)

        d2 = self.up2(d1)
        d2 = self.dec2(d2)

        d3 = self.up3(d2)
        d3 = self.dec3(d3)

        d4 = self.up4(d3)
        d4 = self.dec4(d4)

        return self.outc(d4)

# --- 4. VERİ SETİ (DATASET) ---
class NIRDataset(Dataset):
    def __init__(self, root_path, patch_size=256):
        self.patch_size = patch_size
        self.data_cache = []
        nir_dir = os.path.join(root_path, "NIR")
        rgb_dir = os.path.join(root_path, "RGB")

        nir_files = sorted([f for f in os.listdir(nir_dir) if f.lower().endswith(('.bmp', '.png', '.jpg'))])
        rgb_files = sorted([f for f in os.listdir(rgb_dir) if f.lower().endswith(('.bmp', '.png', '.jpg'))])

        print(f"📥 Veriler RAM'e yükleniyor...")
        for n_file, r_file in tqdm(zip(nir_files, rgb_files), total=len(nir_files)):
            ir_img = cv2.imread(os.path.join(nir_dir, n_file), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
            rgb_img = cv2.imread(os.path.join(rgb_dir, r_file), cv2.IMREAD_COLOR)
            rgb_img = cv2.cvtColor(rgb_img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
            self.data_cache.append({'ir': np.expand_dims(ir_img, axis=-1), 'rgb': rgb_img})

    def __len__(self):
        return len(self.data_cache)

    def __getitem__(self, idx):
        d = self.data_cache[idx]
        ir = torch.from_numpy(d['ir']).permute(2, 0, 1) * 2.0 - 1.0
        rgb = torch.from_numpy(d['rgb']).permute(2, 0, 1) * 2.0 - 1.0
        i, j, h, w = transforms.RandomCrop.get_params(ir, (self.patch_size, self.patch_size))
        return transforms.functional.crop(ir, i, j, h, w), transforms.functional.crop(rgb, i, j, h, w)

# --- 5. PERCEPTUAL LOSS (VGG NORMALİZASYONU EKLENDİ) ---
class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.vgg = models.vgg16(weights='DEFAULT').features[:16].eval().to(DEVICE)
        for p in self.vgg.parameters():
            p.requires_grad = False

        self.criterion = nn.L1Loss()
        self.normalize = Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    def forward(self, fake, real):
        norm_fake = self.normalize((fake + 1.0) / 2.0)
        norm_real = self.normalize((real + 1.0) / 2.0)
        return self.criterion(self.vgg(norm_fake), self.vgg(norm_real))

# --- 6. EĞİTİM KURULUMU VE DÖNGÜSÜ ---
model = VanillaAutoencoder(in_channels=1, out_channels=3).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, betas=(0.5, 0.999))

scaler = torch.amp.GradScaler('cuda')

criterion_pixel = nn.L1Loss()
criterion_vgg = PerceptualLoss()

loader = DataLoader(NIRDataset(ROOT_PATH, PATCH_SIZE), BATCH_SIZE, shuffle=True, pin_memory=True)

print(f"\n🚀 Vanilla Autoencoder Eğitimi A100 Üzerinde Başlıyor...")

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_loss = 0
    loop = tqdm(loader, leave=False)

    for ir, real in loop:
        ir, real = ir.to(DEVICE), real.to(DEVICE)
        optimizer.zero_grad()

        with torch.amp.autocast('cuda', dtype=torch.float16):
            output = model(ir)

            loss_pixel = criterion_pixel(output, real)
            loss_vgg = criterion_vgg(output, real)
            total_loss = loss_pixel + 0.1 * loss_vgg

        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += total_loss.item()
        loop.set_description(f"Epoch [{epoch}/{NUM_EPOCHS}]")
        loop.set_postfix(Loss=total_loss.item())

    if epoch % 100 == 0:
        print(f"Epoch {epoch} | Ortalama Kayıp: {epoch_loss/len(loader):.4f}")

    if epoch % SAVE_EVERY_N_EPOCHS == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
        }, os.path.join(SAVE_PATH, f"VanillaAutoEncoder_epoch_{epoch}.pth"))

print("✅ Eğitim tamamlandı. Drive senkronize ediliyor...")

# RİSK ALMAMAK İÇİN DRIVE'I GÜVENLE ÇIKART
from google.colab import drive
drive.flush_and_unmount()

print("✅ Senkronizasyon başarılı. Oturum kapatılıyor...")
from google.colab import runtime
runtime.unassign()